# Mini Project - 00

### Part 1

In [ ]:
import sys
sys.path.append('..')

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model, should_use_reasoning_model
from IPython.display import Markdown, display

import re
import pandas as pd
from pathlib import Path

In [ ]:
data_path = Path('../data/Sample Messages.txt')
sample_messages = [
    block.strip()
    for block in data_path.read_text(encoding='utf-8').split('\n\n')
    if block.strip()
]

examples = """
Example 1:
News: Breaking News: Kelani River level at 9m."
Intent: Info
District: Colombo
Priority: High
Explanation: The news is providing information about the river level, which is important for public awareness and safety, 
especially in a flood-prone area. It does not call for action but serves to inform the public, hence classified as 'Info' 
with 'High' priority due to potential impact. Since it is a news report, the district is identified as Colombo.

Example 2:
News: "We are trapped on the roof with 3 kids!"
Intent: Rescue
District: None
Priority: High
Explanation: This message is a direct call for help, indicating an urgent need for rescue services. The presence of children
 increases the urgency, making it a 'High' priority. The district is not specified in the message, so it is marked as 'None'.

Example 3:
News: "Donation drive happening at the town hall tomorrow."
Intent: Supply
District: None
Priority: Low
Explanation: This message is about a donation drive, which is an event for gathering supplies. It does not indicate an immediate
need or emergency, so it is classified as 'Supply' with 'Low' priority. The district is not mentioned,
It is classified as 'Info' with 'Low' priority. The district is not mentioned, so it is marked as 'None'.
"""

model = pick_model('openai', 'general')
client = LLMClient('openai', model)

def parse_result(text):
    """Parse 'Intent: X | District: Y | Priority: Z' into a dict."""
    pattern = r'Intent:\s*(\w+)\s*\|\s*District:\s*([^|]+?)\s*\|\s*Priority:\s*(\w+)'
    match = re.search(pattern, text or '')
    if match:
        return {
            'intent': match.group(1).strip(),
            'district': match.group(2).strip(),
            'priority': match.group(3).strip(),
        }
    return {'intent': None, 'district': None, 'priority': None}

records = []
for msg in sample_messages:
    prompt_text, spec = render(
        'few_shot.v1',
        role='news intent, priority and district classifier',
        examples=examples,
        query=f'Review: {msg}',
        constraints='Follow the pattern in examples. Provide the intent, district, priority for your classification. Intent should be one of: Info, Rescue, Supply, or Other. District should be a specific location if mentioned, otherwise None. Priority should be High or Low based on urgency.',
        format='Intent: {{intent}} | District: {{district}} | Priority: {{priority}}'
    )
    llm_messages = [{'role': 'user', 'content': prompt_text}]
    response = client.chat(llm_messages, temperature=0.2)
    parsed = parse_result(response['text'])
    records.append({'message': msg, **parsed})
    print(f"[{parsed['priority']}] {parsed['intent']} | {parsed['district']} — {msg[:60]}")

# Save to CSV
df = pd.DataFrame(records, columns=['message', 'intent', 'district', 'priority'])
output_path = Path('../output/classified_messages.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False, encoding='utf-8')

print(f"\nSaved {len(df)} rows to {output_path}")
df.head()

[High] Info | Colombo — BREAKING: Water levels in Kelani River (Colombo) have reache
[High] Rescue | Gampaha — SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water r
[Low] Info | Kandy — Update: Kandy road cleared near Peradeniya. Traffic moving s
[Low] Supply | Gampaha — Does anyone have extra dry rations for the camp in Gampaha?
[High] Info | Colombo — News just in: Kelani river water level is at 7ft.
[High] Rescue | None — My uncle is stuck in the tree (immediate danger).
[High] Info | Gampaha — Just saw on news that Gampaha town is flooded. Hope everyone
[High] Rescue | None — We are trapped in the attic. 3 kids. Water is entering.
[Low] Supply | None — Donation drive happening at the town hall tomorrow.
[High] Info | None — Report: Heavy rains expected to continue for the next 24 hou
[High] Rescue | Badulla — URGENT: Landslide in Badulla. 2 houses buried. Need rescue t
[Low] Other | None — Can someone recharge my phone? 077-1234567.
[Low] Info | Kelaniya — The Navy has deplo

,message,intent,district,priority
0,BREAKING: Water levels in Kelani River (Colomb...,Info,Colombo,High
1,SOS: 5 people trapped on a roof in Ja-Ela (Gam...,Rescue,Gampaha,High
2,Update: Kandy road cleared near Peradeniya. Tr...,Info,Kandy,Low
3,Does anyone have extra dry rations for the cam...,Supply,Gampaha,Low
4,News just in: Kelani river water level is at 7ft.,Info,Colombo,High


### Part 2


In [14]:
# CoT auto-routes to reasoning model
reasoning_model = pick_model('openai', 'cot')
print(f'Using reasoning model: {reasoning_model}')

client_reasoning = LLMClient('openai', reasoning_model)

# Problem from live class: break time vs travel time confusion
problem = """SCENARIO B: THE GAMPAHA HOSPITAL
Location: Gampaha District General Hospital
Details: "Power has failed in the ICU. We have 3 patients on ventilators with battery backup for 2 hours (Critical). Meanwhile, flood water is entering the ground floor ward where 50 elderly patients are bedridden (High Risk). We have one generator truck arriving in 30 mins, but it can only power one section."
Determine whether rescure boats should reach this location first or not?
"""

# Additional guidance
instruction = """Consider the following factors in your reasoning:
1. If there are some actions taken to rescue people or providing aid, then check whether those actions are sufficient to mitigate the risk.
2. If there are no immediate life threats or the actions taken are sufficient, then rescue boats can prioritize other locations with more urgent needs.
3. If there are immediate life threats that cannot be mitigated with the actions taken, then rescue boats should prioritize this location.
"""

prompt_text, spec = render(
    'cot_reasoning.v1',
    role='rescue boat dispatcher',
    problem=problem
)

# Combine problem with instruction (as done in live class)
full_prompt = f"""text: {prompt_text}

instruction: {instruction}"""

# FIX: Remove space before 'role' to avoid KeyError for Google provider
messages = [{'role': 'user', 'content': full_prompt}]
response = client_reasoning.chat(messages, temperature=1.0, max_tokens=spec.max_tokens)

print('CoT Response:')
print('=' * 80)
print(response['text'])
print('=' * 80)
log_llm_call('openai', reasoning_model, 'cot', response['latency_ms'], response['usage'])

Using reasoning model: o3-mini
CoT Response:
Reasoning Outline:
1. Two issues: ICU with 3 patients on ventilators (battery backup for 2 hours) and the ward with 50 bedridden elderly exposed to incoming flood water.
2. A generator truck arriving in 30 minutes can only power one section.
3. ICU patients have temporary backup support, while the flood impact on the elderly ward presents an immediate, unmitigated risk.
4. The actions available (battery backup and generator truck) are insufficient to protect both sections, leaving the flood-threatened ward vulnerable.

Answer: Rescue boats should be dispatched to this location immediately.


In [5]:
# CoT auto-routes to reasoning model
reasoning_model = pick_model('google', 'cot')
print(f'Using reasoning model: {reasoning_model}')

client_reasoning = LLMClient('google', reasoning_model)

# Problem from live class: break time vs travel time confusion
problem = """A car travels 100 miles in 2 hours. 

Question 1: What is the average speed of the car?
Question 2: If the car stopped for 40 minutes during this 2-hour journey, 
what was the average speed during the actual driving time?

Important: The 2 hours already includes the 40-minute stop."""

# Additional guidance (from live class approach)
instruction = """Solve the following problem step by step.
1. First identify whether the car travelled the entire time without stopping or not.
2. If car stopped for x minutes and overall travelled for y hours, the actual driving duration is y-x.
3. If stopping time x is mentioned, do not add it to the travel duration because it's already included.
   So actual travel time is y (total time) - x (stopping time).
4. If car travelled the entire time without stopping, then average speed is distance / y.
5. If car stopped for x minutes, then average speed during driving is distance / (y - x)."""

prompt_text, spec = render(
    'cot_reasoning.v1',
    role='math tutor',
    problem=problem
)

# Combine problem with instruction (as done in live class)
full_prompt = f"""text: {prompt_text}

instruction: {instruction}"""

messages = [{'role': 'user', 'content': full_prompt}]
response = client_reasoning.chat(messages, temperature=spec.temperature, max_tokens=spec.max_tokens)

print('CoT Response (Travel Time with Break):')
print('=' * 80)
print(response)
print('=' * 80)
log_llm_call('google', reasoning_model, 'cot', response['latency_ms'], response['usage'])

Using reasoning model: gemini-3-pro-preview
CoT Response (Travel Time with Break):
{'text': '**Reasoning Steps:**\n\n1.  **Analyze Question 1 (Overall Average Speed):**\n    *   Identify total distance ($d$) and total time ($y$).\n    *   $d = 100$ miles, $y = 2$ hours.\n    *   Apply formula: $\\text{Average Speed} = \\text{Total Distance} / \\text{Total Time}$.\n\n2.  **Analyze Question 2 (Actual Driving Speed):**\n    *   Identify the stopping time ($x$) included in the total time. $x = 40$ minutes.\n    *   Convert stopping time to hours: $40 \\text{ minutes} = \\frac{40}{60} \\text{ hours} = \\frac{2}{3} \\text{ hours}$.\n    *   Calculate actual driving time ($t_{driving}$) by subtracting stopping time from total time ($y - x$): $2 \\text{ hours} - \\frac{2}{3} \\text{ hours} = \\frac{4}{3} \\text{ hours}$ (or 1 hour 20 minutes).\n    *   Apply formula: $\\text{Driving Speed} = \\text{Total Distance} / \\text{Actual Driving Time}$.\n    *   Calculation: $100 / (\\frac{4}{3})$.\n\

## Part 3: Tree-of-Thought (ToT)

Explore multiple solution paths, then select the best.

In [6]:
problem = """You have a 3-gallon jug and a 5-gallon jug.
How can you measure exactly 4 gallons?"""

prompt_text, spec = render(
    'tot_reasoning.v1',
    role='puzzle solver',
    problem=problem,
    branches='3'
)

messages = [{'role': 'user', 'content': prompt_text}]
response = client_reasoning.chat(messages, temperature=spec.temperature, max_tokens=spec.max_tokens)

print('ToT Response (Multiple Solution Paths):')
print('=' * 80)
print(response)
print('=' * 80)
log_llm_call('openai', reasoning_model, 'tot', response['latency_ms'], response['usage'])

ToT Response (Multiple Solution Paths):
{'text': None, 'usage': {'input_tokens_est': 76, 'context_tokens_est': 0, 'total_est': 79, 'prompt_tokens_actual': 76, 'completion_tokens_actual': 0, 'total_tokens_actual': 76}, 'latency_ms': 43435, 'raw': GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(),
      finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>,
      index=0
    ),
  ],
  model_version='gemini-3-pro-preview',
  response_id='sfdHafbmO8mQmtkP24-AGQ',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    prompt_token_count=76,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=76
      ),
    ],
    thoughts_token_count=4093,
    total_token_count=4169
  )
), 'meta': {'retry_count': 0, 'backoff_ms_total': 0, 'overflow_handled': False}}


## Part 4: Context Overflow Demo

When context exceeds limits, use overflow_summarize prompt.

In [7]:
# Simulate large context
large_context = 'Product details: ' + ' '.join(['Feature description.'] * 500)

prompt_text, spec = render(
    'overflow_summarize.v1',
    context=large_context,
    max_tokens_context='200',
    task='List the top 3 features',
    format='Bullet list'
)

# Use hard_prompt_cap to trigger truncation
client_capped = LLMClient('google', model, hard_prompt_cap=300)
messages = [{'role': 'user', 'content': prompt_text}]

response = client_capped.chat(messages, temperature=0.2)
print('Overflow handled:', response['meta']['overflow_handled'])
print('Response:', response['text'][:200], '...')

Overflow handled: True
Response: The provided context consists of a product description with a long list of feature descriptions. Due to the truncation, the specific features, names, and numbers are unavailable. The summary is theref ...


## Key Takeaways

1. **Zero-shot**: Fast, works for simple tasks
2. **Few-shot**: Examples improve accuracy significantly
3. **CoT/ToT**: Reasoning models required, higher token cost but better logic
4. **Automatic routing**: `pick_model()` selects reasoning models for CoT/ToT
5. **Overflow handling**: Use summarization or truncation when context is too large

**Next:** `04_structured_outputs_json_schema.ipynb`